<a href="https://colab.research.google.com/github/Mike-Umali/NYSI-Capstone-Group-INSMS-/blob/Vector-search/testingv1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### I am using google colab to be able to use SentenceTransformer library in python.

If you also need to use colab, make sure to change the directory to your google drive folder

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Replace the first directory with the actual path of the folder
# Second directory is to rename the lengthy directory to shorter one
# Make sure to avoid using spaces for the folder name in the google drive
!ln -s "/content/drive/MyDrive/ColabNotebooks/CapstoneVector" "/content/capstone_vector"


In [ ]:
import os

# Change the current working directory to your specific folder's shortcut
os.chdir("/content/capstone_vector")

# List files in the directory to verify
print(os.listdir())

# Read a file (example using pandas)
# import pandas as pd
# df = pd.read_csv("supplements_mock_data.csv")
# print(df.head(2))


# Generate Mock Data

### Add more mock data 10 rows (70/30)

In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler

# 1. LOAD THE DATA
# Assuming you have the file 'supplements_mock_data.csv' from the previous step
try:
    df = pd.read_csv("supplements_mock_data.csv")
    print("Loaded CSV file.")
except FileNotFoundError:
    print("Error: CSV not found. Please generate the mock data first.")
    # (If you don't have the file, paste the previous data generation code here)

# 2. INITIALIZE THE AI MODEL
# This ensures the vectors in your DB are created by the SAME brain that will search later.
print("Loading Model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 3. PREPARE DATA FOR VECTORIZATION
# We need to extract the text from the JSON string column
def get_ingredient_text(json_str):
    try:
        data = json.loads(json_str)
        # Join list into a single string: "Whey Protein, Cocoa, Lecithin"
        return ", ".join(data.get("ingredients", []))
    except:
        return ""

print("Extracting text from ingredients...")
df['clean_text'] = df['supplement_ingredient'].apply(get_ingredient_text)

# 4. GENERATE TEXT VECTORS (The "Semantic" Part)
print("Generating text embeddings (this might take a moment)...")
text_vectors = model.encode(df['clean_text'].tolist())

# 5. GENERATE NUTRITION VECTORS (The "Numeric" Part)
# We need to parse the JSON to get the numbers
def get_nutrition_values(json_str):
    try:
        data = json.loads(json_str)
        # We focus on Protein and Carbs for this example, but you can add more
        p = data.get("protein_g", 0)
        c = data.get("carbohydrate_g", 0) # or fat, vit_a, etc
        return [p, c]
    except:
        return [0, 0]

nut_data = df['nutritional_info_per_100g'].apply(get_nutrition_values).tolist()
nut_array = np.array(nut_data)

# Normalize nutrition (0 to 1) so it doesn't overpower the text
scaler = MinMaxScaler()
nut_vectors = scaler.fit_transform(nut_array)

# 6. COMBINE INTO HYBRID VECTOR
# Weight: 70% Text, 30% Nutrition
hybrid_vectors = np.hstack([text_vectors * 0.7, nut_vectors * 0.3])

# 7. SAVE BACK TO DATAFRAME
# We convert the numpy array back to a string so it can be saved in CSV
# Format: "[0.123, -0.456, ...]"
df['vector_100g_ingredient'] = [str(vec.tolist()) for vec in hybrid_vectors]

# For the 'per serving' vector, we will just copy the 100g one for this test
# (In a real app, you would repeat the math above for the serving column)
df['vector_perserving_ingredient'] = df['vector_100g_ingredient']

# Clean up temporary column
df.drop(columns=['clean_text'], inplace=True)

# 8. EXPORT UPDATED FILE
output_filename = "supplements_data_WITH_REAL_VECTORS.csv"
df.to_csv(output_filename, index=False)
print(f"Success! Saved '{output_filename}'.")
print("The columns 'vector_100g_ingredient' now contain REAL AI data.")

### Add more mock data 30 rows (70/30)

In [ ]:
import pandas as pd
import numpy as np
import json
import uuid
import random
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. SETUP & HELPERS
# ==========================================
def get_uuid(): return str(uuid.uuid4())
def get_date(days_ago=0): return (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d %H:%M:%S')

# Dose Form UUIDs (Mock Foreign Keys)
forms = {
    "Powder": "11111111-1111-4111-8111-111111111111",
    "Pill":   "22222222-2222-4222-8222-222222222222",
    "Gel":    "33333333-3333-4333-8333-333333333333",
    "Liquid": "44444444-4444-4444-8444-444444444444",
    "Bar":    "55555555-5555-4555-8555-555555555555",
    "Gummy":  "66666666-6666-4666-8666-666666666666"
}

# ==========================================
# 2. DATA GENERATION (30 ITEMS WITH ALL COLUMNS)
# ==========================================
raw_data = [
    # --- PROTEIN POWDERS ---
    {
        "supplement_name": "Gold Standard 100% Whey", "supplement_brand": "Optimum Nutrition", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "The world's best-selling whey protein powder. Great for muscle support and recovery.",
        "supplement_ingredient": ["Whey Protein Isolate", "Whey Protein Concentrate", "Cocoa", "Lecithin"],
        "nutritional_info_per_100g": {"protein_g": 78, "carbohydrate_g": 5.5, "fat_g": 3.3},
        "nutritional_info_per_serving": {"protein_g": 24, "carbohydrate_g": 1.7, "fat_g": 1.0},
        "nutritional_info_per_serving_definition": "1 Scoop (30.4g)",
        "supplement_warning_label": "Contains Milk and Soy.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Gluten Free", "supplement_website": "https://www.optimumnutrition.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },
    {
        "supplement_name": "Impact Whey Isolate", "supplement_brand": "MyProtein", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Purified whey protein isolate with over 90% protein and low fat.",
        "supplement_ingredient": ["Whey Protein Isolate", "Soy Lecithin", "Flavoring", "Sucralose"],
        "nutritional_info_per_100g": {"protein_g": 82, "carbohydrate_g": 2.5, "fat_g": 0.3},
        "nutritional_info_per_serving": {"protein_g": 21, "carbohydrate_g": 0.6, "fat_g": 0.1},
        "nutritional_info_per_serving_definition": "1 Scoop (25g)",
        "supplement_warning_label": "Contains Milk.", "supplement_certifications": "Labdoor Grade A",
        "supplement_additional_information": "Ranked highest for value.", "supplement_website": "https://www.myprotein.com",
        "batch_testing_org": "Labdoor", "supplement_status": 3
    },
    {
        "supplement_name": "Vegan Protein Blend", "supplement_brand": "Sunwarrior", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "A clean, delicious, plant-based protein for a healthy lifestyle.",
        "supplement_ingredient": ["Pea Protein", "Hemp Protein", "Goji Berry", "Coconut MCTs"],
        "nutritional_info_per_100g": {"protein_g": 75, "carbohydrate_g": 4, "fat_g": 6},
        "nutritional_info_per_serving": {"protein_g": 19, "carbohydrate_g": 2, "fat_g": 2},
        "nutritional_info_per_serving_definition": "1 Scoop (25g)",
        "supplement_warning_label": "Contains Coconut.", "supplement_certifications": "Organic Certified",
        "supplement_additional_information": "Keto Friendly.", "supplement_website": "https://sunwarrior.com",
        "batch_testing_org": None, "supplement_status": 1
    },
    {
        "supplement_name": "Casein Protein", "supplement_brand": "Optimum Nutrition", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Slow digesting protein, best taken before bed for overnight recovery.",
        "supplement_ingredient": ["Micellar Casein", "Cocoa", "Salt", "Gum Blend"],
        "nutritional_info_per_100g": {"protein_g": 73, "carbohydrate_g": 10, "fat_g": 1.5},
        "nutritional_info_per_serving": {"protein_g": 24, "carbohydrate_g": 3, "fat_g": 0.5},
        "nutritional_info_per_serving_definition": "1 Scoop (34g)",
        "supplement_warning_label": "Contains Milk.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Sustained release.", "supplement_website": "https://optimumnutrition.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },
    {
        "supplement_name": "Mass Gainer Extreme", "supplement_brand": "Mutant", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "High calorie mass gainer designed for bodybuilding and hard gainers.",
        "supplement_ingredient": ["Waxy Maize Starch", "Maltodextrin", "Whey Concentrate", "Casein", "MCT Oil"],
        "nutritional_info_per_100g": {"protein_g": 20, "carbohydrate_g": 70, "fat_g": 5},
        "nutritional_info_per_serving": {"protein_g": 56, "carbohydrate_g": 192, "fat_g": 12},
        "nutritional_info_per_serving_definition": "4 Scoops (280g)",
        "supplement_warning_label": "Contains Milk, Soy.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "1100 Calories per serving.", "supplement_website": "https://mutant.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 2
    },

    # --- PILLS & VITAMINS ---
    {
        "supplement_name": "Daily Multivitamin", "supplement_brand": "Nature Made", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Comprehensive daily nutritional support for general wellness.",
        "supplement_ingredient": ["Vitamin A", "Vitamin C", "Vitamin D3", "Zinc Oxide", "Magnesium Oxide"],
        "nutritional_info_per_100g": {"protein_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"vitamin_a_mcg": 750, "vitamin_c_mg": 60, "zinc_mg": 15},
        "nutritional_info_per_serving_definition": "1 Tablet",
        "supplement_warning_label": "Keep out of reach of children.", "supplement_certifications": "USP Verified",
        "supplement_additional_information": "No artificial flavors.", "supplement_website": "https://www.naturemade.com",
        "batch_testing_org": "USP", "supplement_status": 3
    },
    {
        "supplement_name": "Super HD Fat Burner", "supplement_brand": "Cellucor", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "High-definition fat burner and weight loss aid with nootropics.",
        "supplement_ingredient": ["Caffeine Anhydrous", "Green Tea Extract", "Capsimax Cayenne", "N-Acetyl-L-Tyrosine"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"caffeine_mg": 160, "energy_kcal": 0},
        "nutritional_info_per_serving_definition": "1 Capsule",
        "supplement_warning_label": "High Caffeine Content. Not for minors.", "supplement_certifications": None,
        "supplement_additional_information": "Take with water.", "supplement_website": "https://cellucor.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Vitamin D3 5000 IU", "supplement_brand": "Now Foods", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "High potency structural support. Helps maintain strong bones.",
        "supplement_ingredient": ["Vitamin D3 (Cholecalciferol)", "Olive Oil", "Gelatin", "Glycerin"],
        "nutritional_info_per_100g": {"protein_g": 0, "fat_g": 90},
        "nutritional_info_per_serving": {"vitamin_d_iu": 5000, "fat_g": 0.2},
        "nutritional_info_per_serving_definition": "1 Softgel",
        "supplement_warning_label": "Consult doctor if pregnant.", "supplement_certifications": "GMP",
        "supplement_additional_information": "Non-GMO.", "supplement_website": "https://nowfoods.com",
        "batch_testing_org": "UL", "supplement_status": 3
    },
    {
        "supplement_name": "Magnesium Glycinate", "supplement_brand": "Doctor's Best", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "High absorption magnesium that is gentle on the stomach.",
        "supplement_ingredient": ["Magnesium Lysinate Glycinate Chelate", "Microcrystalline Cellulose"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"magnesium_mg": 200},
        "nutritional_info_per_serving_definition": "2 Tablets",
        "supplement_warning_label": "None.", "supplement_certifications": "Non-GMO Verified",
        "supplement_additional_information": "Helps relax muscles.", "supplement_website": "https://drbvitamins.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "B-Complex Plus", "supplement_brand": "Pure Encapsulations", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Balanced B-vitamin formula for energy and nervous system health.",
        "supplement_ingredient": ["Thiamine", "Riboflavin", "Niacin", "Vitamin B6", "Folate", "Vitamin B12", "Biotin"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"vitamin_b12_mcg": 500, "folate_mcg": 400},
        "nutritional_info_per_serving_definition": "1 Capsule",
        "supplement_warning_label": "None.", "supplement_certifications": "Gluten Free",
        "supplement_additional_information": "Hypoallergenic.", "supplement_website": "https://pureencapsulations.com",
        "batch_testing_org": "Internal", "supplement_status": 3
    },
    {
        "supplement_name": "Zinc Picolinate", "supplement_brand": "Thorne", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Highly absorbable zinc for immune function and reproductive health.",
        "supplement_ingredient": ["Zinc Picolinate"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"zinc_mg": 30},
        "nutritional_info_per_serving_definition": "1 Capsule",
        "supplement_warning_label": "Take with food to avoid nausea.", "supplement_certifications": "NSF Sport",
        "supplement_additional_information": "No unnecessary fillers.", "supplement_website": "https://thorne.com",
        "batch_testing_org": "NSF", "supplement_status": 3
    },
    {
        "supplement_name": "Melatonin 5mg", "supplement_brand": "Natrol", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Fast-dissolve tablets to help you fall asleep faster.",
        "supplement_ingredient": ["Melatonin", "Xylitol", "Cellulose Gum", "Strawberry Flavor"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"melatonin_mg": 5},
        "nutritional_info_per_serving_definition": "1 Tablet",
        "supplement_warning_label": "May cause drowsiness. Do not drive.", "supplement_certifications": "Vegetarian",
        "supplement_additional_information": "Drug-free sleep aid.", "supplement_website": "https://natrol.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Iron Complex", "supplement_brand": "Solgar", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "Gentle Iron formulation to support energy utilization.",
        "supplement_ingredient": ["Iron Bisglycinate", "Vitamin C", "Folic Acid", "Vitamin B12"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"iron_mg": 25, "vitamin_c_mg": 100},
        "nutritional_info_per_serving_definition": "1 Vegetable Capsule",
        "supplement_warning_label": "Accidental overdose is fatal to children.", "supplement_certifications": "Kosher, Halal",
        "supplement_additional_information": "Non-constipating.", "supplement_website": "https://solgar.com",
        "batch_testing_org": "Internal", "supplement_status": 3
    },
    {
        "supplement_name": "Hydro Electrolyte Tabs", "supplement_brand": "SIS", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Effervescent tablets designed to replace electrolytes lost through sweat.",
        "supplement_ingredient": ["Citric Acid", "Sodium Bicarbonate", "Sorbitol", "Potassium Chloride"],
        "nutritional_info_per_100g": {"sodium_mg": 8000, "carbohydrate_g": 20},
        "nutritional_info_per_serving": {"sodium_mg": 350, "potassium_mg": 70, "energy_kcal": 9},
        "nutritional_info_per_serving_definition": "1 Tablet (4.5g)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Sport",
        "supplement_additional_information": "Low calorie hydration.", "supplement_website": "https://scienceinsport.com",
        "batch_testing_org": "Informed Sport", "supplement_status": 3
    },

    # --- PRE-WORKOUT & AMINOS ---
    {
        "supplement_name": "Creatine Monohydrate", "supplement_brand": "Thorne", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Pure micronized creatine monohydrate to support athletic performance.",
        "supplement_ingredient": ["Creatine Monohydrate"],
        "nutritional_info_per_100g": {"protein_g": 0, "creatine_g": 100},
        "nutritional_info_per_serving": {"protein_g": 0, "creatine_g": 5},
        "nutritional_info_per_serving_definition": "1 Scoop (5g)",
        "supplement_warning_label": "Drink plenty of water.", "supplement_certifications": "NSF Certified for Sport",
        "supplement_additional_information": "Colorless and odorless.", "supplement_website": "https://thorne.com",
        "batch_testing_org": "NSF", "supplement_status": 3
    },
    {
        "supplement_name": "C4 Original Pre-Workout", "supplement_brand": "Cellucor", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Explosive energy pre-workout for advanced users.",
        "supplement_ingredient": ["Beta-Alanine", "Creatine Nitrate", "Arginine AKG", "Caffeine Anhydrous"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 5},
        "nutritional_info_per_serving": {"caffeine_mg": 150, "beta_alanine_g": 1.6},
        "nutritional_info_per_serving_definition": "1 Scoop (6g)",
        "supplement_warning_label": "Beta-alanine may cause harmless tingling.", "supplement_certifications": None,
        "supplement_additional_information": "America's #1 Selling Pre-workout.", "supplement_website": "https://cellucor.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Pump Surge Stim-Free", "supplement_brand": "Jacked Factory", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Caffeine-free pre-workout pump enhancer.",
        "supplement_ingredient": ["L-Citrulline", "Betaine Anhydrous", "Taurine", "Alpha-GPC"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 5},
        "nutritional_info_per_serving": {"l_citrulline_g": 6, "betaine_g": 2.5},
        "nutritional_info_per_serving_definition": "1 Scoop (15g)",
        "supplement_warning_label": "None.", "supplement_certifications": "GMP",
        "supplement_additional_information": "Nootropic infused.", "supplement_website": "https://jackedfactory.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Total War Pre-Workout", "supplement_brand": "Redcon1", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "High intensity pre-workout with clinical dosages.",
        "supplement_ingredient": ["Citrulline Malate", "Beta-Alanine", "Caffeine Anhydrous", "Juniper Berry Extract"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 2},
        "nutritional_info_per_serving": {"caffeine_mg": 250, "citrulline_g": 6},
        "nutritional_info_per_serving_definition": "1 Scoop (14g)",
        "supplement_warning_label": "Extremely High Caffeine.", "supplement_certifications": None,
        "supplement_additional_information": "Military grade.", "supplement_website": "https://redcon1.com",
        "batch_testing_org": "3rd Party", "supplement_status": 3
    },
    {
        "supplement_name": "Xtend BCAA Original", "supplement_brand": "Scivation", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "Intra-workout BCAA powder for muscle recovery and hydration.",
        "supplement_ingredient": ["L-Leucine", "L-Isoleucine", "L-Valine", "Electrolyte Blend", "Vitamin B6"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0},
        "nutritional_info_per_serving": {"bcaa_g": 7, "sugar_g": 0},
        "nutritional_info_per_serving_definition": "1 Scoop (14g)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Sugar free.", "supplement_website": "https://officialxtend.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },
    {
        "supplement_name": "Glutamine Powder", "supplement_brand": "Optimum Nutrition", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Pure L-Glutamine for post-workout recovery.",
        "supplement_ingredient": ["L-Glutamine"],
        "nutritional_info_per_100g": {"protein_g": 100},
        "nutritional_info_per_serving": {"l_glutamine_g": 5},
        "nutritional_info_per_serving_definition": "1 Teaspoon (5g)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Unflavored, mixes easily.", "supplement_website": "https://optimumnutrition.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },

    # --- BARS, GELS, GUMMIES ---
    {
        "supplement_name": "Oat & Honey Energy Bar", "supplement_brand": "Nature Valley", "supplement_dose_form_id": forms["Bar"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Crunchy granola bars made with whole grain oats and real honey.",
        "supplement_ingredient": ["Whole Grain Oats", "Honey", "Almond Butter", "Sea Salt"],
        "nutritional_info_per_100g": {"protein_g": 10, "carbohydrate_g": 60, "fat_g": 15},
        "nutritional_info_per_serving": {"protein_g": 4, "carbohydrate_g": 25, "fat_g": 6},
        "nutritional_info_per_serving_definition": "1 Bar (42g)",
        "supplement_warning_label": "Contains Almonds.", "supplement_certifications": "Non-GMO",
        "supplement_additional_information": "Great for hiking.", "supplement_website": "https://naturevalley.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Chocolate Peanut Butter Bar", "supplement_brand": "Quest Nutrition", "supplement_dose_form_id": forms["Bar"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "High protein bar with low net carbs and sugar.",
        "supplement_ingredient": ["Milk Protein Isolate", "Peanuts", "Erythritol", "Cocoa"],
        "nutritional_info_per_100g": {"protein_g": 35, "carbohydrate_g": 35, "fat_g": 15},
        "nutritional_info_per_serving": {"protein_g": 20, "carbohydrate_g": 21, "fat_g": 9},
        "nutritional_info_per_serving_definition": "1 Bar (60g)",
        "supplement_warning_label": "Contains Milk, Peanuts.", "supplement_certifications": "Gluten Free Certified",
        "supplement_additional_information": "Low carb option.", "supplement_website": "https://questnutrition.com",
        "batch_testing_org": "Labdoor", "supplement_status": 3
    },
    {
        "supplement_name": "Keto Nut Bar", "supplement_brand": "Adonis", "supplement_dose_form_id": forms["Bar"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "Low carb nut bar designed for the ketogenic diet.",
        "supplement_ingredient": ["Almonds", "Pecans", "Inulin fiber", "Vanilla"],
        "nutritional_info_per_100g": {"protein_g": 12, "carbohydrate_g": 10, "fat_g": 45},
        "nutritional_info_per_serving": {"protein_g": 4, "carbohydrate_g": 3, "fat_g": 15},
        "nutritional_info_per_serving_definition": "1 Bar (35g)",
        "supplement_warning_label": "May contain shell fragments.", "supplement_certifications": "Keto Certified",
        "supplement_additional_information": "High fat, low carb.", "supplement_website": "https://adonis-foods.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Energy Gel Citrus", "supplement_brand": "GU Energy", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Fast-acting energy gel for endurance athletes.",
        "supplement_ingredient": ["Maltodextrin", "Water", "Fructose", "Amino Acids", "Sodium Citrate"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 70},
        "nutritional_info_per_serving": {"carbohydrate_g": 22, "caffeine_mg": 20},
        "nutritional_info_per_serving_definition": "1 Packet (32g)",
        "supplement_warning_label": "None.", "supplement_certifications": "None",
        "supplement_additional_information": "Contains Caffeine.", "supplement_website": "https://guenergy.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Isotonic Energy Gel", "supplement_brand": "SIS", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "World's first isotonic energy gel, no need for water.",
        "supplement_ingredient": ["Water", "Maltodextrin", "Gellan Gum", "Sweetener"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 36},
        "nutritional_info_per_serving": {"carbohydrate_g": 22, "sugar_g": 0.6},
        "nutritional_info_per_serving_definition": "1 Packet (60ml)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Sport",
        "supplement_additional_information": "Easy digestion.", "supplement_website": "https://scienceinsport.com",
        "batch_testing_org": "Informed Sport", "supplement_status": 3
    },
    {
        "supplement_name": "Nordic Omega-3", "supplement_brand": "Nordic Naturals", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Fresh, wild-caught fish oil soft gels.",
        "supplement_ingredient": ["Purified Deep Sea Fish Oil", "Gelatin", "Glycerin", "Natural Lemon Flavor"],
        "nutritional_info_per_100g": {"fat_g": 100, "omega3_g": 60},
        "nutritional_info_per_serving": {"fat_g": 1, "omega3_mg": 690},
        "nutritional_info_per_serving_definition": "2 Softgels",
        "supplement_warning_label": "Contains Fish.", "supplement_certifications": "Friend of the Sea",
        "supplement_additional_information": "Great lemon taste.", "supplement_website": "https://nordic.com",
        "batch_testing_org": "3rd Party Labs", "supplement_status": 3
    },
    {
        "supplement_name": "Turmeric Curcumin", "supplement_brand": "Qunol", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Ultra high absorption turmeric with hydro-soluble technology.",
        "supplement_ingredient": ["Turmeric Root Extract", "Black Pepper Extract", "Gelatin"],
        "nutritional_info_per_100g": {"protein_g": 0},
        "nutritional_info_per_serving": {"turmeric_mg": 1000},
        "nutritional_info_per_serving_definition": "2 Softgels",
        "supplement_warning_label": "None.", "supplement_certifications": "None",
        "supplement_additional_information": "Supports joint health.", "supplement_website": "https://qunol.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Ashwagandha KSM-66", "supplement_brand": "Goli", "supplement_dose_form_id": forms["Gummy"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Delicious gummies to relax, restore and unwind.",
        "supplement_ingredient": ["KSM-66 Ashwagandha Root Extract", "Vitamin D2", "Pectin", "Cane Sugar"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 60},
        "nutritional_info_per_serving": {"ashwagandha_mg": 300, "sugar_g": 2},
        "nutritional_info_per_serving_definition": "1 Gummy",
        "supplement_warning_label": "None.", "supplement_certifications": "Non-GMO",
        "supplement_additional_information": "Vegan friendly.", "supplement_website": "https://goli.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Greens Freak", "supplement_brand": "PharmaFreak", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "High potency greens formula with spirulina and chlorella.",
        "supplement_ingredient": ["Spirulina", "Chlorella", "Alfalfa", "Wheat Grass", "Probiotics"],
        "nutritional_info_per_100g": {"protein_g": 10, "carbohydrate_g": 40},
        "nutritional_info_per_serving": {"superfood_blend_mg": 1500},
        "nutritional_info_per_serving_definition": "1 Scoop",
        "supplement_warning_label": "None.", "supplement_certifications": "GMP",
        "supplement_additional_information": "Daily Detox.", "supplement_website": "https://pharmafreak.com",
        "batch_testing_org": "Internal", "supplement_status": 3
    },
    {
        "supplement_name": "Collagen Peptides", "supplement_brand": "Vital Proteins", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Sourced from grass-fed, pasture-raised bovine hides to ensure a natural, high quality.",
        "supplement_ingredient": ["Bovine Hide Collagen Peptides"],
        "nutritional_info_per_100g": {"protein_g": 90, "fat_g": 0},
        "nutritional_info_per_serving": {"protein_g": 18, "collagen_g": 20},
        "nutritional_info_per_serving_definition": "2 Scoops (20g)",
        "supplement_warning_label": "Kosher.", "supplement_certifications": "Whole30 Approved",
        "supplement_additional_information": "For skin, hair, nails.", "supplement_website": "https://vitalproteins.com",
        "batch_testing_org": "NSF", "supplement_status": 3
    }
]

# ==========================================
# 3. DATAFRAME CREATION & AUDIT FIELDS
# ==========================================
df = pd.DataFrame(raw_data)

# Add Primary Key
df['id'] = [get_uuid() for _ in range(len(df))]

# Add Audit Columns (Required by Diagram)
df['created_on'] = get_date(days_ago=30)
df['created_by'] = "system_seed_script"
df['last_modified_on'] = get_date(days_ago=1)
df['last_modified_by'] = "human_reviewer"

# Convert JSON/List fields to string JSON for CSV storage
for col in ['supplement_ingredient', 'nutritional_info_per_100g', 'nutritional_info_per_serving']:
    df[col] = df[col].apply(json.dumps)

print(f"Dataframe created with {len(df)} rows. All Schema columns present.")

# ==========================================
# 4. VECTORIZATION (Strictly Typed)
# ==========================================
print("Generating Vectors using AI...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# Helper: Extract clean text from JSON string
def clean_text(json_str):
    try:
        data = json.loads(json_str)
        # Handle if it's a list (ingredients) or dict (nutrition) - though ingredients is list here
        if isinstance(data, list): return ", ".join(data)
        if isinstance(data, dict): return str(data)
        return ""
    except: return ""

# A. Text Vectors
text_inputs = df['supplement_ingredient'].apply(clean_text).tolist()
text_vectors = model.encode(text_inputs)

# B. Nutrition Vectors (Protein + Carbs Normalized)
def get_nut_values(json_str):
    try:
        data = json.loads(json_str)
        return [data.get('protein_g', 0), data.get('carbohydrate_g', 0)]
    except: return [0,0]

nut_array = np.array(df['nutritional_info_per_100g'].apply(get_nut_values).tolist())
scaler = MinMaxScaler()
nut_vectors = scaler.fit_transform(nut_array)

# C. Hybrid Vectors
hybrid_vectors = np.hstack([text_vectors * 0.7, nut_vectors * 0.3])

# Save Vectors
df['vector_100g_ingredient'] = [str(vec.tolist()) for vec in hybrid_vectors]
df['vector_perserving_ingredient'] = df['vector_100g_ingredient'] # Using same logic for mock

# ==========================================
# 5. COLUMN REORDERING (Match Diagram)
# ==========================================
schema_order = [
    'id', 'supplement_dose_form_id', 'supplement_input_type', 'human_in_the_loop',
    'supplement_name', 'supplement_brand', 'supplement_description',
    'supplement_ingredient', 'nutritional_info_per_100g', 'nutritional_info_per_serving',
    'nutritional_info_per_serving_definition', 'supplement_warning_label',
    'supplement_certifications', 'supplement_additional_information',
    'supplement_website', 'batch_testing_org', 'supplement_status',
    'vector_100g_ingredient', 'vector_perserving_ingredient',
    'created_on', 'created_by', 'last_modified_on', 'last_modified_by'
]

# Ensure we have exactly these columns
df = df[schema_order]

# ==========================================
# 6. EXPORT
# ==========================================
filename = "supplements_full_schema.csv"
df.to_csv(filename, index=False)
print(f"Success! Generated '{filename}' matching the exact database schema.")

### Add more mock data 30 rows (50/50)

In [ ]:
import pandas as pd
import numpy as np
import json
import uuid
import random
from datetime import datetime, timedelta
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 1. SETUP & HELPERS
# ==========================================
def get_uuid(): return str(uuid.uuid4())
def get_date(days_ago=0): return (datetime.now() - timedelta(days=days_ago)).strftime('%Y-%m-%d %H:%M:%S')

forms = {
    "Powder": "11111111-1111-4111-8111-111111111111",
    "Pill":   "22222222-2222-4222-8222-222222222222",
    "Gel":    "33333333-3333-4333-8333-333333333333",
    "Liquid": "44444444-4444-4444-8444-444444444444",
    "Bar":    "55555555-5555-4555-8555-555555555555",
    "Gummy":  "66666666-6666-4666-8666-666666666666"
}

# ==========================================
# 2. DATA GENERATION (30 ITEMS)
# ==========================================
raw_data = [
    # --- PROTEIN POWDERS ---
    {
        "supplement_name": "Gold Standard 100% Whey", "supplement_brand": "Optimum Nutrition", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "The world's best-selling whey protein powder. Great for muscle support and recovery.",
        "supplement_ingredient": ["Whey Protein Isolate", "Whey Protein Concentrate", "Cocoa", "Lecithin"],
        "nutritional_info_per_100g": {"protein_g": 78, "carbohydrate_g": 5.5, "fat_g": 3.3},
        "nutritional_info_per_serving": {"protein_g": 24, "carbohydrate_g": 1.7, "fat_g": 1.0},
        "nutritional_info_per_serving_definition": "1 Scoop (30.4g)",
        "supplement_warning_label": "Contains Milk and Soy.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Gluten Free", "supplement_website": "https://www.optimumnutrition.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },
    {
        "supplement_name": "Impact Whey Isolate", "supplement_brand": "MyProtein", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Purified whey protein isolate with over 90% protein and low fat.",
        "supplement_ingredient": ["Whey Protein Isolate", "Soy Lecithin", "Flavoring", "Sucralose"],
        "nutritional_info_per_100g": {"protein_g": 82, "carbohydrate_g": 2.5, "fat_g": 0.3},
        "nutritional_info_per_serving": {"protein_g": 21, "carbohydrate_g": 0.6, "fat_g": 0.1},
        "nutritional_info_per_serving_definition": "1 Scoop (25g)",
        "supplement_warning_label": "Contains Milk.", "supplement_certifications": "Labdoor Grade A",
        "supplement_additional_information": "Ranked highest for value.", "supplement_website": "https://www.myprotein.com",
        "batch_testing_org": "Labdoor", "supplement_status": 3
    },
    {
        "supplement_name": "Vegan Protein Blend", "supplement_brand": "Sunwarrior", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "A clean, delicious, plant-based protein for a healthy lifestyle.",
        "supplement_ingredient": ["Pea Protein", "Hemp Protein", "Goji Berry", "Coconut MCTs"],
        "nutritional_info_per_100g": {"protein_g": 75, "carbohydrate_g": 4, "fat_g": 6},
        "nutritional_info_per_serving": {"protein_g": 19, "carbohydrate_g": 2, "fat_g": 2},
        "nutritional_info_per_serving_definition": "1 Scoop (25g)",
        "supplement_warning_label": "Contains Coconut.", "supplement_certifications": "Organic Certified",
        "supplement_additional_information": "Keto Friendly.", "supplement_website": "https://sunwarrior.com",
        "batch_testing_org": None, "supplement_status": 1
    },
    {
        "supplement_name": "Casein Protein", "supplement_brand": "Optimum Nutrition", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Slow digesting protein, best taken before bed for overnight recovery.",
        "supplement_ingredient": ["Micellar Casein", "Cocoa", "Salt", "Gum Blend"],
        "nutritional_info_per_100g": {"protein_g": 73, "carbohydrate_g": 10, "fat_g": 1.5},
        "nutritional_info_per_serving": {"protein_g": 24, "carbohydrate_g": 3, "fat_g": 0.5},
        "nutritional_info_per_serving_definition": "1 Scoop (34g)",
        "supplement_warning_label": "Contains Milk.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Sustained release.", "supplement_website": "https://optimumnutrition.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },
    {
        "supplement_name": "Mass Gainer Extreme", "supplement_brand": "Mutant", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "High calorie mass gainer designed for bodybuilding and hard gainers.",
        "supplement_ingredient": ["Waxy Maize Starch", "Maltodextrin", "Whey Concentrate", "Casein", "MCT Oil"],
        "nutritional_info_per_100g": {"protein_g": 20, "carbohydrate_g": 70, "fat_g": 5},
        "nutritional_info_per_serving": {"protein_g": 56, "carbohydrate_g": 192, "fat_g": 12},
        "nutritional_info_per_serving_definition": "4 Scoops (280g)",
        "supplement_warning_label": "Contains Milk, Soy.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "1100 Calories per serving.", "supplement_website": "https://mutant.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 2
    },

    # --- PILLS & VITAMINS ---
    {
        "supplement_name": "Daily Multivitamin", "supplement_brand": "Nature Made", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Comprehensive daily nutritional support for general wellness.",
        "supplement_ingredient": ["Vitamin A", "Vitamin C", "Vitamin D3", "Zinc Oxide", "Magnesium Oxide"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"vitamin_a_mcg": 750, "vitamin_c_mg": 60, "zinc_mg": 15},
        "nutritional_info_per_serving_definition": "1 Tablet",
        "supplement_warning_label": "Keep out of reach of children.", "supplement_certifications": "USP Verified",
        "supplement_additional_information": "No artificial flavors.", "supplement_website": "https://www.naturemade.com",
        "batch_testing_org": "USP", "supplement_status": 3
    },
    {
        "supplement_name": "Super HD Fat Burner", "supplement_brand": "Cellucor", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "High-definition fat burner and weight loss aid with nootropics.",
        "supplement_ingredient": ["Caffeine Anhydrous", "Green Tea Extract", "Capsimax Cayenne", "N-Acetyl-L-Tyrosine"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"caffeine_mg": 160, "energy_kcal": 0},
        "nutritional_info_per_serving_definition": "1 Capsule",
        "supplement_warning_label": "High Caffeine Content. Not for minors.", "supplement_certifications": None,
        "supplement_additional_information": "Take with water.", "supplement_website": "https://cellucor.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Vitamin D3 5000 IU", "supplement_brand": "Now Foods", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "High potency structural support. Helps maintain strong bones.",
        "supplement_ingredient": ["Vitamin D3 (Cholecalciferol)", "Olive Oil", "Gelatin", "Glycerin"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 90},
        "nutritional_info_per_serving": {"vitamin_d_iu": 5000, "fat_g": 0.2},
        "nutritional_info_per_serving_definition": "1 Softgel",
        "supplement_warning_label": "Consult doctor if pregnant.", "supplement_certifications": "GMP",
        "supplement_additional_information": "Non-GMO.", "supplement_website": "https://nowfoods.com",
        "batch_testing_org": "UL", "supplement_status": 3
    },
    {
        "supplement_name": "Magnesium Glycinate", "supplement_brand": "Doctor's Best", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "High absorption magnesium that is gentle on the stomach.",
        "supplement_ingredient": ["Magnesium Lysinate Glycinate Chelate", "Microcrystalline Cellulose"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"magnesium_mg": 200},
        "nutritional_info_per_serving_definition": "2 Tablets",
        "supplement_warning_label": "None.", "supplement_certifications": "Non-GMO Verified",
        "supplement_additional_information": "Helps relax muscles.", "supplement_website": "https://drbvitamins.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "B-Complex Plus", "supplement_brand": "Pure Encapsulations", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Balanced B-vitamin formula for energy and nervous system health.",
        "supplement_ingredient": ["Thiamine", "Riboflavin", "Niacin", "Vitamin B6", "Folate", "Vitamin B12", "Biotin"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"vitamin_b12_mcg": 500, "folate_mcg": 400},
        "nutritional_info_per_serving_definition": "1 Capsule",
        "supplement_warning_label": "None.", "supplement_certifications": "Gluten Free",
        "supplement_additional_information": "Hypoallergenic.", "supplement_website": "https://pureencapsulations.com",
        "batch_testing_org": "Internal", "supplement_status": 3
    },
    {
        "supplement_name": "Zinc Picolinate", "supplement_brand": "Thorne", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Highly absorbable zinc for immune function and reproductive health.",
        "supplement_ingredient": ["Zinc Picolinate"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"zinc_mg": 30},
        "nutritional_info_per_serving_definition": "1 Capsule",
        "supplement_warning_label": "Take with food to avoid nausea.", "supplement_certifications": "NSF Sport",
        "supplement_additional_information": "No unnecessary fillers.", "supplement_website": "https://thorne.com",
        "batch_testing_org": "NSF", "supplement_status": 3
    },
    {
        "supplement_name": "Melatonin 5mg", "supplement_brand": "Natrol", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Fast-dissolve tablets to help you fall asleep faster.",
        "supplement_ingredient": ["Melatonin", "Xylitol", "Cellulose Gum", "Strawberry Flavor"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"melatonin_mg": 5},
        "nutritional_info_per_serving_definition": "1 Tablet",
        "supplement_warning_label": "May cause drowsiness. Do not drive.", "supplement_certifications": "Vegetarian",
        "supplement_additional_information": "Drug-free sleep aid.", "supplement_website": "https://natrol.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Iron Complex", "supplement_brand": "Solgar", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "Gentle Iron formulation to support energy utilization.",
        "supplement_ingredient": ["Iron Bisglycinate", "Vitamin C", "Folic Acid", "Vitamin B12"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"iron_mg": 25, "vitamin_c_mg": 100},
        "nutritional_info_per_serving_definition": "1 Vegetable Capsule",
        "supplement_warning_label": "Accidental overdose is fatal to children.", "supplement_certifications": "Kosher, Halal",
        "supplement_additional_information": "Non-constipating.", "supplement_website": "https://solgar.com",
        "batch_testing_org": "Internal", "supplement_status": 3
    },
    {
        "supplement_name": "Hydro Electrolyte Tabs", "supplement_brand": "SIS", "supplement_dose_form_id": forms["Pill"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Effervescent tablets designed to replace electrolytes lost through sweat.",
        "supplement_ingredient": ["Citric Acid", "Sodium Bicarbonate", "Sorbitol", "Potassium Chloride"],
        "nutritional_info_per_100g": {"sodium_mg": 8000, "carbohydrate_g": 20, "fat_g": 0},
        "nutritional_info_per_serving": {"sodium_mg": 350, "potassium_mg": 70, "energy_kcal": 9},
        "nutritional_info_per_serving_definition": "1 Tablet (4.5g)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Sport",
        "supplement_additional_information": "Low calorie hydration.", "supplement_website": "https://scienceinsport.com",
        "batch_testing_org": "Informed Sport", "supplement_status": 3
    },

    # --- PRE-WORKOUT & AMINOS ---
    {
        "supplement_name": "Creatine Monohydrate", "supplement_brand": "Thorne", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Pure micronized creatine monohydrate to support athletic performance.",
        "supplement_ingredient": ["Creatine Monohydrate"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0, "creatine_g": 100},
        "nutritional_info_per_serving": {"protein_g": 0, "creatine_g": 5},
        "nutritional_info_per_serving_definition": "1 Scoop (5g)",
        "supplement_warning_label": "Drink plenty of water.", "supplement_certifications": "NSF Certified for Sport",
        "supplement_additional_information": "Colorless and odorless.", "supplement_website": "https://thorne.com",
        "batch_testing_org": "NSF", "supplement_status": 3
    },
    {
        "supplement_name": "C4 Original Pre-Workout", "supplement_brand": "Cellucor", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Explosive energy pre-workout for advanced users.",
        "supplement_ingredient": ["Beta-Alanine", "Creatine Nitrate", "Arginine AKG", "Caffeine Anhydrous"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 5, "fat_g": 0},
        "nutritional_info_per_serving": {"caffeine_mg": 150, "beta_alanine_g": 1.6},
        "nutritional_info_per_serving_definition": "1 Scoop (6g)",
        "supplement_warning_label": "Beta-alanine may cause harmless tingling.", "supplement_certifications": None,
        "supplement_additional_information": "America's #1 Selling Pre-workout.", "supplement_website": "https://cellucor.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Pump Surge Stim-Free", "supplement_brand": "Jacked Factory", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Caffeine-free pre-workout pump enhancer.",
        "supplement_ingredient": ["L-Citrulline", "Betaine Anhydrous", "Taurine", "Alpha-GPC"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 5, "fat_g": 0},
        "nutritional_info_per_serving": {"l_citrulline_g": 6, "betaine_g": 2.5},
        "nutritional_info_per_serving_definition": "1 Scoop (15g)",
        "supplement_warning_label": "None.", "supplement_certifications": "GMP",
        "supplement_additional_information": "Nootropic infused.", "supplement_website": "https://jackedfactory.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Total War Pre-Workout", "supplement_brand": "Redcon1", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "High intensity pre-workout with clinical dosages.",
        "supplement_ingredient": ["Citrulline Malate", "Beta-Alanine", "Caffeine Anhydrous", "Juniper Berry Extract"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 2, "fat_g": 0},
        "nutritional_info_per_serving": {"caffeine_mg": 250, "citrulline_g": 6},
        "nutritional_info_per_serving_definition": "1 Scoop (14g)",
        "supplement_warning_label": "Extremely High Caffeine.", "supplement_certifications": None,
        "supplement_additional_information": "Military grade.", "supplement_website": "https://redcon1.com",
        "batch_testing_org": "3rd Party", "supplement_status": 3
    },
    {
        "supplement_name": "Xtend BCAA Original", "supplement_brand": "Scivation", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "Intra-workout BCAA powder for muscle recovery and hydration.",
        "supplement_ingredient": ["L-Leucine", "L-Isoleucine", "L-Valine", "Electrolyte Blend", "Vitamin B6"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"bcaa_g": 7, "sugar_g": 0},
        "nutritional_info_per_serving_definition": "1 Scoop (14g)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Sugar free.", "supplement_website": "https://officialxtend.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },
    {
        "supplement_name": "Glutamine Powder", "supplement_brand": "Optimum Nutrition", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Pure L-Glutamine for post-workout recovery.",
        "supplement_ingredient": ["L-Glutamine"],
        "nutritional_info_per_100g": {"protein_g": 100, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"l_glutamine_g": 5},
        "nutritional_info_per_serving_definition": "1 Teaspoon (5g)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Choice",
        "supplement_additional_information": "Unflavored, mixes easily.", "supplement_website": "https://optimumnutrition.com",
        "batch_testing_org": "Informed Choice", "supplement_status": 3
    },

    # --- BARS, GELS, GUMMIES ---
    {
        "supplement_name": "Oat & Honey Energy Bar", "supplement_brand": "Nature Valley", "supplement_dose_form_id": forms["Bar"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Crunchy granola bars made with whole grain oats and real honey.",
        "supplement_ingredient": ["Whole Grain Oats", "Honey", "Almond Butter", "Sea Salt"],
        "nutritional_info_per_100g": {"protein_g": 10, "carbohydrate_g": 60, "fat_g": 15},
        "nutritional_info_per_serving": {"protein_g": 4, "carbohydrate_g": 25, "fat_g": 6},
        "nutritional_info_per_serving_definition": "1 Bar (42g)",
        "supplement_warning_label": "Contains Almonds.", "supplement_certifications": "Non-GMO",
        "supplement_additional_information": "Great for hiking.", "supplement_website": "https://naturevalley.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Chocolate Peanut Butter Bar", "supplement_brand": "Quest Nutrition", "supplement_dose_form_id": forms["Bar"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "High protein bar with low net carbs and sugar.",
        "supplement_ingredient": ["Milk Protein Isolate", "Peanuts", "Erythritol", "Cocoa"],
        "nutritional_info_per_100g": {"protein_g": 35, "carbohydrate_g": 35, "fat_g": 15},
        "nutritional_info_per_serving": {"protein_g": 20, "carbohydrate_g": 21, "fat_g": 9},
        "nutritional_info_per_serving_definition": "1 Bar (60g)",
        "supplement_warning_label": "Contains Milk, Peanuts.", "supplement_certifications": "Gluten Free Certified",
        "supplement_additional_information": "Low carb option.", "supplement_website": "https://questnutrition.com",
        "batch_testing_org": "Labdoor", "supplement_status": 3
    },
    {
        "supplement_name": "Keto Nut Bar", "supplement_brand": "Adonis", "supplement_dose_form_id": forms["Bar"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "Low carb nut bar designed for the ketogenic diet.",
        "supplement_ingredient": ["Almonds", "Pecans", "Inulin fiber", "Vanilla"],
        "nutritional_info_per_100g": {"protein_g": 12, "carbohydrate_g": 10, "fat_g": 45},
        "nutritional_info_per_serving": {"protein_g": 4, "carbohydrate_g": 3, "fat_g": 15},
        "nutritional_info_per_serving_definition": "1 Bar (35g)",
        "supplement_warning_label": "May contain shell fragments.", "supplement_certifications": "Keto Certified",
        "supplement_additional_information": "High fat, low carb.", "supplement_website": "https://adonis-foods.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Energy Gel Citrus", "supplement_brand": "GU Energy", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Fast-acting energy gel for endurance athletes.",
        "supplement_ingredient": ["Maltodextrin", "Water", "Fructose", "Amino Acids", "Sodium Citrate"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 70, "fat_g": 0},
        "nutritional_info_per_serving": {"carbohydrate_g": 22, "caffeine_mg": 20},
        "nutritional_info_per_serving_definition": "1 Packet (32g)",
        "supplement_warning_label": "None.", "supplement_certifications": "None",
        "supplement_additional_information": "Contains Caffeine.", "supplement_website": "https://guenergy.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Isotonic Energy Gel", "supplement_brand": "SIS", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "World's first isotonic energy gel, no need for water.",
        "supplement_ingredient": ["Water", "Maltodextrin", "Gellan Gum", "Sweetener"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 36, "fat_g": 0},
        "nutritional_info_per_serving": {"carbohydrate_g": 22, "sugar_g": 0.6},
        "nutritional_info_per_serving_definition": "1 Packet (60ml)",
        "supplement_warning_label": "None.", "supplement_certifications": "Informed Sport",
        "supplement_additional_information": "Easy digestion.", "supplement_website": "https://scienceinsport.com",
        "batch_testing_org": "Informed Sport", "supplement_status": 3
    },
    {
        "supplement_name": "Nordic Omega-3", "supplement_brand": "Nordic Naturals", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "webscraper", "human_in_the_loop": False,
        "supplement_description": "Fresh, wild-caught fish oil soft gels.",
        "supplement_ingredient": ["Purified Deep Sea Fish Oil", "Gelatin", "Glycerin", "Natural Lemon Flavor"],
        "nutritional_info_per_100g": {"fat_g": 100, "omega3_g": 60, "protein_g": 0, "carbohydrate_g": 0},
        "nutritional_info_per_serving": {"fat_g": 1, "omega3_mg": 690},
        "nutritional_info_per_serving_definition": "2 Softgels",
        "supplement_warning_label": "Contains Fish.", "supplement_certifications": "Friend of the Sea",
        "supplement_additional_information": "Great lemon taste.", "supplement_website": "https://nordic.com",
        "batch_testing_org": "3rd Party Labs", "supplement_status": 3
    },
    {
        "supplement_name": "Turmeric Curcumin", "supplement_brand": "Qunol", "supplement_dose_form_id": forms["Gel"],
        "supplement_input_type": "manual", "human_in_the_loop": True,
        "supplement_description": "Ultra high absorption turmeric with hydro-soluble technology.",
        "supplement_ingredient": ["Turmeric Root Extract", "Black Pepper Extract", "Gelatin"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 0, "fat_g": 0},
        "nutritional_info_per_serving": {"turmeric_mg": 1000},
        "nutritional_info_per_serving_definition": "2 Softgels",
        "supplement_warning_label": "None.", "supplement_certifications": "None",
        "supplement_additional_information": "Supports joint health.", "supplement_website": "https://qunol.com",
        "batch_testing_org": None, "supplement_status": 3
    },
    {
        "supplement_name": "Ashwagandha KSM-66", "supplement_brand": "Goli", "supplement_dose_form_id": forms["Gummy"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Delicious gummies to relax, restore and unwind.",
        "supplement_ingredient": ["KSM-66 Ashwagandha Root Extract", "Vitamin D2", "Pectin", "Cane Sugar"],
        "nutritional_info_per_100g": {"protein_g": 0, "carbohydrate_g": 60, "fat_g": 0},
        "nutritional_info_per_serving": {"ashwagandha_mg": 300, "sugar_g": 2},
        "nutritional_info_per_serving_definition": "1 Gummy",
        "supplement_warning_label": "None.", "supplement_certifications": "Non-GMO",
        "supplement_additional_information": "Vegan friendly.", "supplement_website": "https://goli.com",
        "batch_testing_org": None, "supplement_status": 2
    },
    {
        "supplement_name": "Greens Freak", "supplement_brand": "PharmaFreak", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "manual", "human_in_the_loop": False,
        "supplement_description": "High potency greens formula with spirulina and chlorella.",
        "supplement_ingredient": ["Spirulina", "Chlorella", "Alfalfa", "Wheat Grass", "Probiotics"],
        "nutritional_info_per_100g": {"protein_g": 10, "carbohydrate_g": 40, "fat_g": 0},
        "nutritional_info_per_serving": {"superfood_blend_mg": 1500},
        "nutritional_info_per_serving_definition": "1 Scoop",
        "supplement_warning_label": "None.", "supplement_certifications": "GMP",
        "supplement_additional_information": "Daily Detox.", "supplement_website": "https://pharmafreak.com",
        "batch_testing_org": "Internal", "supplement_status": 3
    },
    {
        "supplement_name": "Collagen Peptides", "supplement_brand": "Vital Proteins", "supplement_dose_form_id": forms["Powder"],
        "supplement_input_type": "webscraper", "human_in_the_loop": True,
        "supplement_description": "Sourced from grass-fed, pasture-raised bovine hides to ensure a natural, high quality.",
        "supplement_ingredient": ["Bovine Hide Collagen Peptides"],
        "nutritional_info_per_100g": {"protein_g": 90, "fat_g": 0, "carbohydrate_g": 0},
        "nutritional_info_per_serving": {"protein_g": 18, "collagen_g": 20},
        "nutritional_info_per_serving_definition": "2 Scoops (20g)",
        "supplement_warning_label": "Kosher.", "supplement_certifications": "Whole30 Approved",
        "supplement_additional_information": "For skin, hair, nails.", "supplement_website": "https://vitalproteins.com",
        "batch_testing_org": "NSF", "supplement_status": 3
    }
]

# ==========================================
# 3. DATAFRAME CREATION
# ==========================================
df = pd.DataFrame(raw_data)
df['id'] = [get_uuid() for _ in range(len(df))]
df['created_on'] = get_date(days_ago=30)
df['created_by'] = "system_seed_script"
df['last_modified_on'] = get_date(days_ago=1)
df['last_modified_by'] = "human_reviewer"

for col in ['supplement_ingredient', 'nutritional_info_per_100g', 'nutritional_info_per_serving']:
    df[col] = df[col].apply(json.dumps)

print(f"Dataframe created with {len(df)} rows. All Schema columns present.")

# ==========================================
# 4. VECTORIZATION (Strictly Typed)
# ==========================================
print("Generating Vectors using AI...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# --- A. RICH TEXT VECTORIZATION (Crucial Fix) ---
# We combine Name + Brand + Description + Ingredients to create a full picture.
def create_rich_text(row):
    try:
        # Extract ingredients from JSON
        ing_data = json.loads(row['supplement_ingredient'])
        ing_list = ing_data if isinstance(ing_data, list) else ing_data.get('ingredients', [])
        ing_str = ", ".join(ing_list)

        # Combine all text fields
        # "Gold Standard Optimum Nutrition The world's best... Whey Protein, Cocoa"
        return f"{row['supplement_name']} {row['supplement_brand']} {row['supplement_description']} {ing_str}"
    except:
        return ""

print("Generating Rich Text Embeddings...")
df['rich_text'] = df.apply(create_rich_text, axis=1)
text_vectors = model.encode(df['rich_text'].tolist())

# --- B. NUTRITION VECTORIZATION (Added Fat) ---
# Now we track Protein, Carbs, AND Fat
def get_nut_values(json_str):
    try:
        data = json.loads(json_str)
        return [
            data.get('protein_g', 0),
            data.get('carbohydrate_g', 0),
            data.get('fat_g', 0) # Added Fat
        ]
    except: return [0,0,0]

nut_array = np.array(df['nutritional_info_per_100g'].apply(get_nut_values).tolist())
scaler = MinMaxScaler()
nut_vectors = scaler.fit_transform(nut_array)

# --- C. HYBRID VECTORIZATION (50/50 Split) ---
# 50% Text, 50% Nutrition
hybrid_vectors = np.hstack([text_vectors * 0.5, nut_vectors * 0.5])

# Save Vectors
df['vector_100g_ingredient'] = [str(vec.tolist()) for vec in hybrid_vectors]
df['vector_perserving_ingredient'] = df['vector_100g_ingredient']

# Cleanup temp columns
df.drop(columns=['rich_text'], inplace=True)

# ==========================================
# 5. COLUMN REORDERING
# ==========================================
schema_order = [
    'id', 'supplement_dose_form_id', 'supplement_input_type', 'human_in_the_loop',
    'supplement_name', 'supplement_brand', 'supplement_description',
    'supplement_ingredient', 'nutritional_info_per_100g', 'nutritional_info_per_serving',
    'nutritional_info_per_serving_definition', 'supplement_warning_label',
    'supplement_certifications', 'supplement_additional_information',
    'supplement_website', 'batch_testing_org', 'supplement_status',
    'vector_100g_ingredient', 'vector_perserving_ingredient',
    'created_on', 'created_by', 'last_modified_on', 'last_modified_by'
]

df = df[schema_order]

# ==========================================
# 6. EXPORT
# ==========================================
filename = "supplements_full_schema_balanced_v2.csv"
df.to_csv(filename, index=False)
print(f"Success! Generated '{filename}' matching the exact database schema.")

# Vector and Similarity Search

## Test 0

I am using scikit-learn for vectorisation currently (statistical method, need hard-code), since I have issue in installing sentence-transformers (deep learning method) due to OS error

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# === 1. SETUP MOCK DATA ===
# We create a small databse of supplements
# NB: Retinol is Vitamin A, Ascorbic Acid is Vitamin C, Tocopherol is Vitamin E

data = [
    {
        "id": 101,
        "name": "Nutrition Pure Vitamin A Tabs",
        "description": "High potency Vitamin A supplement to support eye health and immune function.",
        "form": "Pill",
        "ingredients": "Vitamin A Acetate, Corn Starch, Gelatin",
        "benefits": "Supports vision, immune system, and skin health.",
        "brand": "Nutrition Pure",
        "vit_a_mcg": 900,
        "vit_c_mg": 0,
        "protein_g": 0
    },
    {
        "id": 102,
        "name": "Vision Health Gummies",
        "description": "Vitamin A gummies designed to enhance eye health and provide antioxidant benefits.",
        "form": "Gummy",
        "ingredients": "Retinol Palmitate, Glucose Syrup, Pectin",
        "benefits": "Enhances vision and provides antioxidant support.",
        "brand": "Wellness",
        "vit_a_mcg": 800,
        "vit_c_mg": 10,
        "protein_g": 0
    },
    {
        "id": 103,
        "name": "Nordic Cod Liver Oil",
        "description": "Rich source of Vitamin A and D from cod liver oil to support overall health.",
        "form": "Liquid",
        "ingredients": "Wild caught cod liver oil, Omega-3 fatty acids, Retinol",
        "benefits": "Supports immune function, bone health, and vision.",
        "brand": "HealthPlus",
        "vit_a_mcg": 250,
        "vit_c_mg": 0,
        "protein_g": 0
    },
    {
        "id": 104,
        "name": "Super C-500",
        "description": "Effervescent Vitamin C tablets to support immune health and skin vitality.",
        "form": "Pill",
        "ingredients": "Ascorbic Acid, Rose Hips, Citrus Bioflavonoids",
        "benefits": "Boosts immune system and promotes healthy skin.",
        "brand": "VitaHealth",
        "vit_a_mcg": 0,
        "vit_c_mg": 500,
        "protein_g": 0
    },
    {
        "id": 105,
        "name": "Mega Whey Protein Shake",
        "description": "High-protein shake mix to support muscle recovery and energy levels.",
        "form": "Powder",
        "ingredients": "Whey Protein Isolate, Cocoa Powder, Stevia",
        "benefits": "Supports muscle recovery and provides sustained energy.",
        "brand": "FitLife",
        "vit_a_mcg": 0,
        "vit_c_mg": 0,
        "protein_g": 25
    }
]

## Test 1

In [ ]:
df = pd.DataFrame(data)

# === 2. VECTORISATION (The logic) ===

# 1. normalise the nutritional values to a 0-1 scale
scaler = MinMaxScaler()
nutritional_features = df[['vit_a_mcg', 'vit_c_mg', 'protein_g']].values
normalized_nutrition = scaler.fit_transform(nutritional_features)

# 2. text vectorization using TF-IDF
# this converts text data into numerical vectors based on word frequency
vectorizer = TfidfVectorizer(stop_words='english')
text_embeddings = vectorizer.fit_transform(df['ingredients']).toarray()

# 3. hybrid embedding creation (combining text and nutritional data)
# 50% weight to text embeddings and 50% to nutritional data
weighted_nutrition = normalized_nutrition * 0.5
hybrid_vectors = np.hstack([text_embeddings, weighted_nutrition])

print("Hybrid Vectors Shape:", hybrid_vectors.shape)
print("Hybrid Vectors Sample:\n", hybrid_vectors[:2])

In [ ]:
# === 3. SIMILARITY CALCULATION ===
# Function to get top N similar supplements based on hybrid vectors
def find_alternatives(target_id, database, vectors):
    # find index of target supplement
    target_idx = database[database['id'] == target_id].index[0]

    # get the target vector
    target_vector = vectors[target_idx].reshape(1, -1)

    # compute cosine similarity between target and all others
    scores = cosine_similarity(target_vector, vectors)[0]

    # store results in a DataFrame
    results = database.copy()
    results['similarity_score'] = scores

    # filter out the target supplement itself and sort by similarity score
    return results[results['id'] != target_id].sort_values(by='similarity_score', ascending=False)

In [ ]:
# === 4. TESTING THE FUNCTION ===
# Example: Find alternatives for supplement with ID 101

target_supplement_id = 101
print(f"\nUser is viewing: {df.loc[0, 'name']}")
print("Finding alternatives...\n")

matces = find_alternatives(target_supplement_id, df, hybrid_vectors)

print("---RECOMMENDED ALTERNATIVES---")
for i, row in matces.iterrows():
    print(f"Name: {row['name']}")
    print(f"Description: {row['description']}")
    print(f"Form: {row['form']}")
    print(f"Ingredients: {row['ingredients']}")
    print(f"Benefits: {row['benefits']}")
    print(f"Brand: {row['brand']}")
    print(f"Similarity Score: {row['similarity_score']:.4f}")
    print("------------------------------")

## Test 2

In [ ]:
# === 2. TRAIN THE VECTORIZER (The "Dictionary") ===

# 1. Normalise the database
scaler2 = MinMaxScaler()
db_nutrition = scaler.fit_transform(df[['vit_a_mcg', 'vit_c_mg', 'protein_g']])

# 2. tf-idf for DB (text)
vectorizer2 = TfidfVectorizer(stop_words='english')
db_text_vectors = vectorizer.fit_transform(df['ingredients']).toarray()

# 3. create db hybrid vec
# weight: text counts for 50%, nutrition 50%
db_hybrid = np.hstack([db_text_vectors* 0.5, db_nutrition * 0.5])

print("System ready. Database vectorised")

In [ ]:
# === 3. SEARCH FUNCTION ===

def search_by_text(user_query, db_hybrid_vectors, vectorizer, dataframe, top_k=3):
    # step 1: process user text
    # we use .transform(), NOT .fit_transform()
    # this maps teh user's words to the existing database columns
    query_text_vector = vectorizer.transform([user_query]).toarray()

    # step 2: handle missing nutrition
    # the user typed text, so they didn't provide "900mg" numbers
    # we create a dummy nutrition vector of [0,0,0] - [vit_a_mcg, vit_c_mg, protein_g]
    # this ensures the vector shapes match (text cols + 3 nutrition cols)
    query_nutrition = np.array([[0,0,0]])

    # step 3: create query hybrid vector
    query_hybrid = np.hstack([query_text_vector*0.5, query_nutrition*0.5])

    # step 4: cosine similarity
    scores = cosine_similarity(query_hybrid, db_hybrid_vectors)[0]

    # return results
    results = dataframe.copy()
    results['score'] = scores
    return results.sort_values('score', ascending=False).head(top_k)

In [ ]:
# === 4. TEST THE SEARCH FUNCTION ===

# user_input = "Vitamin C"
user_input = input("\nEnter your search query for supplements: ")

print(f"\nUser search input: '{user_input}'\n")
matches = search_by_text(user_input, db_hybrid, vectorizer, df, top_k=3)

print("---SEARCH RESULTS---")
for i, row in matches.iterrows():
    if row['score'] > 0:
        print(f"Name: {row['name']}")
        print(f"Description: {row['description']}")
        print(f"Form: {row['form']}")
        print(f"Ingredients: {row['ingredients']}")
        print(f"Benefits: {row['benefits']}")
        print(f"Brand: {row['brand']}")
        print(f"Score: {row['score']:.4f}")
        print("------------------------------")
    else:
        print("No relevant results found.")
        break

## Test 3

The pip installs the AI model. You only need to do this once per session

In [ ]:
# !pip install sentence-transformers
!pip install cosine_similarity

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# === 1. SETUP DATABASE (MOCK DATA) ===
data = [
    # Note: I am NOT adding "Vitamin A" to the text here.
    # The AI will figure out the connection on its own.
    {"id": 101, "name": "Pure Vit A Tabs", "ingredients": "Vitamin A Acetate, Gelatin", "vit_a": 900, "vit_c": 0},
    {"id": 102, "name": "Vision Health Gummies", "ingredients": "Retinol Palmitate, Glucose, Pectin", "vit_a": 800, "vit_c": 10},
    {"id": 103, "name": "Nordic Cod Liver Oil", "ingredients": "Wild Caught Cod Liver Oil, Omega-3", "vit_a": 250, "vit_c": 0},
    {"id": 104, "name": "Super C-500", "ingredients": "Ascorbic Acid, Citrus Bioflavonoids", "vit_a": 0, "vit_c": 500},
    {"id": 105, "name": "Zinc Plus", "ingredients": "Zinc Gluconate, Magnesium", "vit_a": 0, "vit_c": 0}
]

df = pd.DataFrame(data)

In [ ]:
# === 2. LOAD AI MODEL & VECTORIZE DB ===
print("Loading AI Model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# A. Vectorize text (ingredients)
# this creates a 384-dim vector for each product
db_text_vec = model.encode(df['ingredients'].tolist())

# B. Normalize nutrition
scaler3 = MinMaxScaler()
db_nutrition = scaler3.fit_transform(df[['vit_a', 'vit_c']])

# C. Combine (hybrid vector)
# we weight text 50% and nutrition 50%
# db_hybrid = np.hstack([db_text_vec * 0.5, db_nutrition * 0.5])
db_hybrid = np.hstack([db_text_vec * 0.4, db_nutrition * 0.6])

print("Database Vectorised successfully")

In [ ]:
# === 3. SEARCH FUNCTION ===

def search_by_user_text(user_query, target_vit_a, target_vit_c, db_vectors, model, scaler, dataframe):
  print(f"\nProcessing Query: '{user_query}'...")

  # A. Vectorize Text
  text_vec = model.encode([user_query])

  # B. Vectorize Nutrition (Crucial Step!)
  # We must scale the USER'S numbers using the SAME scaler as the database
  # Example: If user wants 900mcg, and max is 1000, this becomes 0.9
  raw_nutrition = np.array([[target_vit_a, target_vit_c]])
  user_nutrition_vec = scaler3.transform(raw_nutrition)

  # C. Combine
  # user_hybrid = np.hstack([text_vec * 0.5, user_nutrition_vec * 0.5])
  user_hybrid = np.hstack([text_vec * 0.4, user_nutrition_vec * 0.6])

  # D. Search
  scores = cosine_similarity(user_hybrid, db_vectors)[0]

  results = dataframe.copy()
  results['score'] = scores
  return results.sort_values('score', ascending=False)

In [ ]:
# === 4. TEST SCENARIO ===
user_text_query = input("\nEnter you search query for supplement: ")

user_vit_a_value = input("\nEnter the amount of vitamin a (number only): ")
user_vit_c_value = input("\nEnter the amount of vitamin c (number only): ")


print(f"\nSearching for: '{user_text_query} with {user_vit_a_value}mg vit A and/or {user_vit_c_value}mg vit C'\n")

matches = search_by_user_text(
    user_query=user_text_query,
    target_vit_a=user_vit_a_value,
    target_vit_c=user_vit_c_value,
    db_vectors=db_hybrid,
    model=model,
    scaler=scaler3,
    dataframe=df
    )

print("---SEARCH RESULTS---")
for i, row in matches.iterrows():
    if row['score'] > 0:
        print(f"Name: {row['name']}")
        # print(f"Description: {row['description']}")
        # print(f"Form: {row['form']}")
        print(f"Ingredients: {row['ingredients']}")
        # print(f"Benefits: {row['benefits']}")
        # print(f"Brand: {row['brand']}")
        print(f"Score: {row['score']:.4f}")
        print("------------------------------")
    else:
        print("No relevant results found.")
        break

## Test 4

In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# 1. LOAD DATA WITH REAL VECTORS
# filename = "supplements_data_WITH_REAL_VECTORS.csv"
filename = "supplements_full_schema.csv"
df = pd.read_csv(filename)

# convert the string vectors back to actual number arrays
# this is required because CSVs store everything as text
print("converting CSV text back to arrays...")
df['vector_object'] = df['vector_100g_ingredient'].apply(json.loads)

# stack them into a matrix for the math to work
db_vectors = np.vstack(df['vector_object'].values)

In [ ]:
# 2. SETUP THE MODEL (MUST BE SAME MODEL AS IN THE MOCK DATA)
# MOCK DATA VECTOR IS ALSO USING SENTENCE TRANSFORMERS

model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# 3. SEARCH FUNCTION
def find_supplement(user_query, target_nutrition=[0,0]):
  print(f"\nSearching for: '{user_query}...'")

  # A. vectorise user query
  text_vec = model.encode([user_query])

  # B. handle nutrition (user input needs to be scaled conceptually)
  # for this simple test, we assume user input is alr somewhat normalised
  # or we just use 0s if they didnt specify numbers
  nut_vec = np.array([target_nutrition])

  # C. combine (use same weights as mock data)
  query_hybrid = np.hstack([text_vec * 0.7, nut_vec * 0.3])

  # D. similarity calc
  scores = cosine_similarity(query_hybrid, db_vectors)[0]
  scores_percent = scores*100

  # E. show results
  results = df.copy()
  results['score'] = scores_percent
  return results.sort_values('score', ascending=False).head(5)

In [ ]:
# 4. TEST IT

user_input = input("\nEnter your search query for supplement: ")
# print(f"\nSearching for: '{user_input}'\n")

query = find_supplement(user_input)

print("\n--- RESULTS ---")
for i, row in query.iterrows():
  print(f"Name: {row['supplement_name']}")
  print(f"Brand: {row['supplement_brand']}")
  print(f"Match Score: {row['score']:.3f}%")
  print("-" * 30)

## Test 5

In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ==========================================
# 1. LOAD DATA & INITIALIZE MODEL
# ==========================================
# Load the dataset generated in the previous step
try:
    df = pd.read_csv("supplements_full_schema_balanced_v2.csv")
    print(f"Loaded Data: {len(df)} rows")
except FileNotFoundError:
    print("Error: CSV file not found. Please run the data generation script first.")
    exit()

print("Loading AI Model (all-MiniLM-L6-v2)...")
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================

# --- A. RICH TEXT EMBEDDING ---
# Combine Name + Brand + Description + Ingredients for maximum context
def create_rich_text(row):
    try:
        # Parse ingredients from JSON string
        ing_data = json.loads(row['supplement_ingredient'])
        # Handle list vs dict structure
        ing_list = ing_data if isinstance(ing_data, list) else ing_data.get('ingredients', [])
        ing_str = ", ".join(ing_list)

        # Construct the "Super String"
        # format: "[Name] [Brand] [Description] [Ingredients]"
        return f"{row['supplement_name']} {row['supplement_brand']} {row['supplement_description']} {ing_str}"
    except Exception as e:
        return ""

print("Generating Rich Text Embeddings...")
df['rich_text'] = df.apply(create_rich_text, axis=1)
text_vectors = model.encode(df['rich_text'].tolist())

# --- B. NUTRITION NORMALIZATION ---
# Extract Protein, Carbs, AND Fat
def get_macros(json_str):
    try:
        data = json.loads(json_str)
        return [
            data.get('protein_g', 0),
            data.get('carbohydrate_g', 0),
            data.get('fat_g', 0)
        ]
    except:
        return [0, 0, 0]

# Extract and Normalize
macro_raw = np.array(df['nutritional_info_per_100g'].apply(get_macros).tolist())
scaler = MinMaxScaler()
macro_vectors = scaler.fit_transform(macro_raw)

# --- C. HYBRID VECTOR STORAGE ---
# We store the full hybrid vector (Text + Nutrition) for the Hybrid Search case
# Text (384 dims) + Nutrition (3 dims) = 387 dims
hybrid_vectors = np.hstack([text_vectors * 0.5, macro_vectors * 0.5])

print("Vectorization Complete.\n")

In [ ]:
# ==========================================
# 3. THE SMART SEARCH ENGINE
# ==========================================

def find_supplements(user_query, target_macros=None, top_k=5):
    """
    user_query: str (e.g., "Oat Bar")
    target_macros: list [Protein, Carbs, Fat] (e.g., [20, 10, 5]) or None
    """
    print(f"Searching for: '{user_query}'...")

    # 1. Vectorize the User's Text Query
    query_text_vec = model.encode([user_query]) # Shape (1, 384)

    # 2. DYNAMIC SEARCH LOGIC
    if target_macros is None:
        # --- MODE A: TEXT-ONLY SEARCH ---
        # User did NOT specify nutrition targets.
        # We only compare against the text portion of the data to avoid the "Zero Penalty".

        print("-> Mode: Text-Only (No nutrition targets specified)")
        scores = cosine_similarity(query_text_vec, text_vectors)[0]

    else:
        # --- MODE B: HYBRID SEARCH ---
        # User specified nutrition targets.
        # We compare Text (50%) + Nutrition (50%)

        print(f"-> Mode: Hybrid (Target Macros: {target_macros})")

        # Prepare User Macro Vector
        # IMPORTANT: We must transform user input using the SAME scaler as the DB
        user_macro_raw = np.array([target_macros])
        user_macro_vec = scaler.transform(user_macro_raw)

        # Combine (0.5 weight for each)
        query_hybrid_vec = np.hstack([query_text_vec * 0.5, user_macro_vec * 0.5])

        # Calculate Similarity against DB Hybrid Vectors
        scores = cosine_similarity(query_hybrid_vec, hybrid_vectors)[0]

    # 3. FORMAT RESULTS
    results = df.copy()
    results['match_score'] = scores * 100 # Convert to percentage

    # Sort and return
    return results.sort_values('match_score', ascending=False).head(top_k)

In [ ]:
# ==========================================
# 4. TEST SCENARIOS
# ==========================================

# SCENARIO 1: Pure Text Search (Should match Name/Brand perfectly now)
print("--- TEST 1: Exact Name Match ---")
case_A = input("\nEnter your search query for supplement: ")
matches = find_supplements(case_A)
for i, row in matches.iterrows():
    print(f"> {row['match_score']:.1f}% | {row['supplement_name']} | ({row['supplement_brand']})")

print("\n" + "="*40 + "\n")

# SCENARIO 2: Hybrid Search (High Protein Requirement)
# "I want a chocolate bar with around 20g protein"
print("--- TEST 2: Hybrid (Text + High Protein) ---")
# Note: Macros are [Protein, Carbs, Fat]. We aim for [20, 20, 10]
matches = find_supplements("Protein powder", target_macros=[80, 20, 5])
for i, row in matches.iterrows():
    print(f"> {row['match_score']:.1f}% | {row['supplement_name']} | ({row['supplement_brand']})")

## Test 6

In [ ]:
import pandas as pd
import numpy as np
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# ==========================================
# 1. LOAD DATA & INITIALIZE MODEL
# ==========================================
try:
    df = pd.read_csv("supplements_full_schema_balanced_v2.csv")
    print(f"Loaded Database: {len(df)} products")
except FileNotFoundError:
    print("Error: CSV file not found. Please run the generation script first.")
    exit()

print("Loading AI Model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
# ==========================================
# 2. FEATURE ENGINEERING (DB SETUP)
# ==========================================

# A. RICH TEXT (Name + Brand + Description + Ingredients)
def create_rich_text(row):
    try:
        ing_data = json.loads(row['supplement_ingredient'])
        ing_list = ing_data if isinstance(ing_data, list) else ing_data.get('ingredients', [])
        ing_str = ", ".join(ing_list)
        return f"{row['supplement_name']} {row['supplement_brand']} {row['supplement_description']} {ing_str}"
    except: return ""

df['rich_text'] = df.apply(create_rich_text, axis=1)
text_vectors = model.encode(df['rich_text'].tolist())

# B. NUTRITION (Protein, Carbs, Fat)
def get_macros(json_str):
    try:
        data = json.loads(json_str)
        return [data.get('protein_g', 0), data.get('carbohydrate_g', 0), data.get('fat_g', 0)]
    except: return [0, 0, 0]

macro_raw = np.array(df['nutritional_info_per_100g'].apply(get_macros).tolist())
scaler = MinMaxScaler()
macro_vectors = scaler.fit_transform(macro_raw)

# C. HYBRID VECTORS (50% Text, 50% Nutrition)
hybrid_vectors = np.hstack([text_vectors * 0.5, macro_vectors * 0.5])

In [ ]:
# ==========================================
# 3. SEARCH ENGINE FUNCTION
# ==========================================
def find_supplements(query_text, target_macros=None, top_k=3):
    # 1. Vectorize Text
    query_text_vec = model.encode([query_text])

    # 2. Determine Search Mode
    if target_macros is None:
        # MODE: Text-Only (Front Label Scan)
        # We slice the DB vectors to compare ONLY text columns (0 to 384)
        print(f"   [System] Running Text-Only Search for: '{query_text}'")
        scores = cosine_similarity(query_text_vec, text_vectors)[0]
    else:
        # MODE: Hybrid (Back Label Scan)
        # We compare Text + Nutrition
        print(f"   [System] Running Hybrid Search for: '{query_text}' + Macros {target_macros}")

        # Scale User Macros using the DB Scaler
        user_macro_vec = scaler.transform(np.array([target_macros]))

        # Create User Hybrid Vector
        query_hybrid = np.hstack([query_text_vec * 0.5, user_macro_vec * 0.5])

        # Calculate Similarity
        scores = cosine_similarity(query_hybrid, hybrid_vectors)[0]

    # 3. Return Results
    results = df.copy()
    results['match_score'] = scores * 100
    return results.sort_values('match_score', ascending=False).head(top_k)

In [ ]:
# ==========================================
# 4. TEST SCENARIOS (USE CASE A & B)
# ==========================================

print("\n" + "="*50)
print("TEST SCENARIO 1: USE CASE A (Finding Alternatives)")
print("User searches for an existing product to find alternatives.")
print("="*50)

# User Input: "Gold Standard Whey"
# Expectation: Should find itself (100%) and then similar items like MyProtein
case_a = input("\nEnter your search query for supplement: ")
results = find_supplements(case_a)
# results = find_supplements("Gold Standard 100% Whey")
for i, row in results.iterrows():
    print(f"> {row['match_score']:.1f}% | {row['supplement_name']} ({row['supplement_brand']}) - {row['supplement_description'][:30]}...")


print("\n" + "="*50)
print("TEST SCENARIO 2: USE CASE B (OCR - External Product)")
print("User scans a product NOT in our database.")
print("="*50)

# --- SUB-CASE 2.1: FRONT LABEL SCAN (Brand/Name) ---
# Imagine the user scanned a "Ghost Lifestyle Legend Pre-Workout".
# This product is NOT in our mock DB.
# The AI Agent extracts the text "Ghost Legend Pre-Workout V2" from the image.

ai_agent_output_text = "Ghost Legend Pre-Workout V2"
ai_agent_output_macros = None # Front label usually has no macros

print(f"\n--- 2.1 OCR Scan: Front Label (Brand Only) ---")
print(f"   [Input] OCR extracted: '{ai_agent_output_text}'")

results = find_supplements(ai_agent_output_text, ai_agent_output_macros)

for i, row in results.iterrows():
    # Expectation: Should match "Total War" or "C4" or "Pump Surge"
    print(f"> {row['match_score']:.1f}% | {row['supplement_name']} ({row['supplement_brand']}) - {row['supplement_description'][:30]}...")


# --- SUB-CASE 2.2: BACK LABEL SCAN (Ingredients + Nutrition) ---
# Imagine the user scanned "Orgain Organic Plant Protein".
# This is NOT in our DB.
# The AI Agent extracts ingredients and the macro table.

# Extracted from Image:
ai_agent_output_text = "Organic Pea Protein, Brown Rice Protein, Chia Seed"
ai_agent_output_macros = [21, 15, 4] # [Protein, Carbs, Fat]

print(f"\n--- 2.2 OCR Scan: Back Label (Ingredients + Nutrition) ---")
print(f"   [Input] Text: '{ai_agent_output_text}'")
print(f"   [Input] Macros: {ai_agent_output_macros}")

results = find_supplements(ai_agent_output_text, ai_agent_output_macros)

for i, row in results.iterrows():
    # Expectation: Should match "Vegan Protein Blend" (Sunwarrior) because of Pea Protein + Macros
    print(f"> {row['match_score']:.1f}% | {row['supplement_name']} ({row['supplement_brand']}) - {row['supplement_description'][:30]}......")